In [8]:
# pip install pystac-client planetary-computer stackstac xarray numpy matplotlib imageio rasterio python-dateutil

import os
from datetime import datetime
import matplotlib.pyplot as plt
import imageio.v3 as iio
from pystac_client import Client
import planetary_computer as pc
import stackstac
import numpy as np
from dateutil.rrule import rrule, MONTHLY
from rasterio.enums import Resampling

# PARAMETRY
CLOUD_COVER_MAX = 10
BANDS = ["B04", "B03", "B02"]
EPSG = 4326
RES_DEG = 0.0002
OUT_DIR = "out_anim"
os.makedirs(OUT_DIR, exist_ok=True)
BBOX = [120.94, 14.47, 121.01, 14.58]  # Manila Bay

# Kwartały od 2020 do teraz
quarter_starts = list(rrule(freq=MONTHLY, interval=3, dtstart=datetime(2020, 1, 1), until=datetime.now()))

def fetch_best_item(client, bbox, start, end):
    """Pobierz pierwszą sensowną scenę (z danymi, bez chmur)."""
    search = client.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{start.date().isoformat()}/{end.date().isoformat()}",
        query={"eo:cloud_cover": {"lt": CLOUD_COVER_MAX}},
        sortby=[{"field": "properties.datetime", "direction": "asc"}],
        max_items=10,
    )
    items = list(search.get_items())
    for item in items:
        try:
            stack = stackstac.stack([item], assets=["B08"], bounds_latlon=bbox,
                                    epsg=EPSG, resolution=RES_DEG,
                                    resampling=Resampling.nearest).astype("float32")
            nir = stack.sel(band="B08").isel(time=0).values
            if not np.all(np.isnan(nir)):
                return item
        except Exception:
            continue
    return None

def render_rgb_image(item):
    """Zbuduj RGB obraz z itemu Sentinel-2."""
    stack = stackstac.stack([item], assets=BANDS, bounds_latlon=BBOX,
                            epsg=EPSG, resolution=RES_DEG,
                            resampling=Resampling.nearest).astype("float32")
    rgb = stack.sel(band=BANDS) / 10000.0
    rgb = rgb.clip(0, 1)
    img = np.stack([rgb.sel(band=b).isel(time=0).values for b in BANDS], axis=-1)
    img = np.squeeze(img)  # 💡 usuń wymiar time
    extent = [float(stack.x.min()), float(stack.x.max()), float(stack.y.min()), float(stack.y.max())]
    return img, extent

# Główne wykonanie
client = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)
frames = []

for i in range(len(quarter_starts) - 1):
    start = quarter_starts[i]
    end = quarter_starts[i + 1]
    item = fetch_best_item(client, BBOX, start, end)
    if item is None:
        continue

    img, extent = render_rgb_image(item)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img, extent=extent)
    ax.set_title(f"Manila Bay — {start.date().isoformat()}")
    ax.axis('off')

    frame_path = os.path.join(OUT_DIR, f"frame_{i:02d}.png")
    fig.savefig(frame_path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    frames.append(frame_path)

# Tworzenie GIF
gif_path = os.path.join(OUT_DIR, "manila_bay_rgb_timelapse.gif")
iio.imwrite(gif_path, [iio.imread(f) for f in frames], duration=0.8)
print(f"✅ GIF zapisany: {gif_path}")


c:\Users\Airly\anaconda3\envs\GeoPython_Ukraine2024_v2\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
c:\Users\Airly\anaconda3\envs\GeoPython_Ukraine2024_v2\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
c:\Users\Airly\anaconda3\envs\GeoPython_Ukraine2024_v2\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
c:\Users\Airly\anaconda3\envs\GeoPython_Ukraine2024_v2\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
c:\Users\Airly\anaconda3\envs\GeoPython_Ukraine2024_v2\lib\site-packages\pystac_client\item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
c:\Users\Airly\anaconda3\envs\GeoPython_Ukraine2024_v2\lib\site-packag

✅ GIF zapisany: out_anim\manila_bay_rgb_timelapse.gif


In [2]:
# air pollution

In [3]:
# rozwoj miast


In [4]:
# amazonia

In [5]:
# soja